# 🎯 HunterBot + Gemini: Cuaderno de Análisis Inteligente de Mercado

Este cuaderno conecta **Google Gemini** con el motor de scraping y búsqueda en tiempo real de **HunterBot** (`/api_search`).

### ¿Qué permite este cuaderno?
1. **Búsqueda en vivo**: Consulta los scrapers de *CosasDeBarcos, TopBarcos, Boat24, Pisos.com, Fotocasa, Chollometro, Idealo y Amazon*.
2. **Function Calling con Gemini**: Gemini decide cuándo y cómo invocar los scrapers según tu pregunta en lenguaje natural.
3. **Análisis profundo**: Tablas comparativas con Pandas, gráficos de dispersión (Precio vs Eslora / m²) y recomendaciones de compra personalizadas.

In [ ]:
# 1. Instalar librerías necesarias
!pip install -q google-genai pandas matplotlib seaborn requests

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

# Endpoint público de HunterBot en Firebase Cloud Functions
HUNTERBOT_API_URL = "https://us-central1-hunterbot-app.cloudfunctions.net/api_search"

# Obtener API Key de Gemini desde los secretos de Colab o variable de entorno
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

def buscar_hunterbot(query: str, category: str = "other", location: str = None, price_max: float = None) -> list:
    """Invoca los scrapers especializados de HunterBot y devuelve las ofertas en formato JSON."""
    payload = {
        "query": query,
        "category": category,
        "location": location,
        "price_max": price_max
    }
    try:
        r = requests.post(HUNTERBOT_API_URL, json=payload, timeout=40)
        if r.status_code == 200:
            data = r.json()
            return data.get("results", [])
        else:
            print(f"Error HTTP {r.status_code}: {r.text[:200]}")
            return []
    except Exception as e:
        print(f"Error conectando a HunterBot API: {e}")
        return []

print("✅ Conector HunterBot listo.")

## 🚤 Ejemplo 1: Búsqueda y Análisis de Barcos con Pandas

In [ ]:
# Realizar una búsqueda directa en portales náuticos
resultados_barcos = buscar_hunterbot("lanchas zar formenti de ocasion", category="boat")

df_barcos = pd.DataFrame(resultados_barcos)
if not df_barcos.empty:
    cols = [c for c in ["title", "price", "length_m", "year_built", "provider", "url"] if c in df_barcos.columns]
    display(df_barcos[cols].sort_values(by="price", ascending=False))
else:
    print("No se obtuvieron resultados para la búsqueda.")

## 🧠 Ejemplo 2: Asesoramiento Inteligente con Gemini AI

In [ ]:
from google import genai

if not GEMINI_API_KEY:
    print("⚠️ Por favor define tu GEMINI_API_KEY en los secretos de Colab (icono de la llave a la izquierda).")
else:
    client = genai.Client(api_key=GEMINI_API_KEY)

    def analizar_con_gemini(pregunta_usuario: str, categoria: str = "boat"):
        """Busca con los scrapers de HunterBot y hace que Gemini analice las ofertas en profundidad."""
        print(f"🔎 Rastreando portales para: '{pregunta_usuario}'...")
        ofertas = buscar_hunterbot(pregunta_usuario, category=categoria)
        
        if not ofertas:
            print("No se encontraron ofertas activas en los portales.")
            return
        
        prompt = f"""
        Eres un asesor técnico y de inversiones experto en el mercado español.
        El cliente pregunta: '{pregunta_usuario}'.
        
        Aquí tienes las ofertas reales obtenidas en directo de los portales:
        {json.dumps(ofertas[:10], ensure_ascii=False, indent=2)}
        
        Por favor, genera un informe en español con:
        1. Tabla comparativa clara con las mejores opciones (modelo, precio, portal y enlace).
        2. Análisis de cuál ofrece la mejor relación calidad-precio y por qué.
        3. Aspectos técnicos y legales críticos a revisar antes de la compra.
        4. Rango de precio recomendado para negociar con el vendedor.
        """
        
        response = client.models.generate_content(
            model="gemma-4-26b-a4b-it",
            contents=prompt
        )
        
        print("\n=== RECOMENDACIÓN Y DICTAMEN DEL ASESOR GEMINI ===\n")
        print(response.text)

    # Probar el análisis asistido con Gemini
    analizar_con_gemini("Zar Formenti 53 o 57 de ocasion", categoria="boat")

## 🏠 Ejemplo 3: Análisis Inmobiliario / Terrenos

In [ ]:
if GEMINI_API_KEY:
    analizar_con_gemini("Terrenos edificables o parcelas rusticas en Foz", categoria="real_estate")